# Interaction-energy figures — DOPC/secondary-lipid systems

Reproduces the five figure types from *Effects of Lipid Headgroups on the Mechanical
Properties and In Vitro Cellular Internalization of Liposomes* (Langmuir 2025), adapted to
the present systems:

| | this work |
|---|---|
| compositions | 65, 72, 80, 87, 95 mol% DOPC |
| secondary lipids | DOPA, DOPG, DLPA, DLPG |
| ensemble | NPT (no applied surface tension) |

**Figures produced**
1. `Fig_KA_vs_partial.png` — K_A vs partial (i–j) energy: (a) all systems, (b) per composition + R²
2. `Fig_KA_vs_total.png` — K_A vs total energy: (a) all systems, (b) per composition + R²
3. `Fig_decomposed_partial.png` — 4-panel decomposed **partial** energies
4. `Fig_decomposed_total.png` — 4-panel decomposed **total** energies
5. `Fig_ion_lipid.png` — ion–headgroup / ion–phosphate energies (solid vs hatched)

Colours are the same lipid palette used in the K_A, order-parameter, RDF and Voronoi figures.


## 1 · Setup

In [ ]:
# !pip install numpy matplotlib scipy
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
try:
    from scipy import stats
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
print("scipy:", HAVE_SCIPY)

## 2 · Style — consistent with all other figures

Lipid hue is fixed across every figure in the paper. Marker shape distinguishes lipid in
scatter plots (as in the published Fig. 4/5 style).

In [ ]:
COMPS  = [65, 72, 80, 87, 95]                 # mol % DOPC
LIPIDS = ["DOPG", "DOPA", "DLPG", "DLPA"]     # secondary lipids

COLORS = {"DOPG": "#E8000B",   # red
          "DOPA": "#1F4E79",   # navy
          "DLPG": "#4C9A2A",   # green
          "DLPA": "#E0A500",   # gold
          "DOPC": "#2C7FB8"}   # steel blue (pure-DOPC reference)
MARKERS = {"DOPG": "o", "DOPA": "s", "DLPG": "^", "DLPA": "D"}

plt.rcParams.update({
    "font.size": 11, "axes.labelsize": 13, "xtick.labelsize": 11,
    "ytick.labelsize": 11, "legend.fontsize": 10,
    "axes.linewidth": 1.0, "figure.dpi": 110,
})

# pure DOPC reference (previous study)
KA_PURE_DOPC = (245.81, 9.85)     # mN/m

## 3 · Data

`KA` is filled in with the measured values. The energy dictionaries are **placeholders** —
replace them with your rerun output. Sign/scale convention follows the paper:
energies in **×1000 kcal/mol** (partial energies mostly negative, total energies positive).

GROMACS reports kJ/mol, so convert with `kj_to_kcal_k()` below.

In [ ]:
def kj_to_kcal_k(e_kJ):
    """GROMACS kJ/mol  ->  x1000 kcal/mol (paper units)."""
    return np.asarray(e_kJ, float) / 4.184 / 1000.0

# --- K_A (mN/m), measured ---
KA = {
    "DOPA": [246.4, 250.5, 228.9, 261.8, 247.1],
    "DOPG": [236.7, 223.3, 217.3, 217.0, 260.0],
    "DLPG": [216.2, 209.3, 274.1, 246.0, 287.3],
    "DLPA": [224.9, 269.7, 221.0, 264.8, 262.0],
}
KA_ERR = {
    "DOPA": [13.0, 38.5, 14.4, 12.1, 17.2],
    "DOPG": [35.9, 12.1, 14.9, 13.5, 20.5],
    "DLPG": [23.6, 14.4, 16.5, 12.4, 30.3],
    "DLPA": [20.7, 12.8, 16.7, 16.0, 14.7],
}

# --- PARTIAL (i-j) interaction energies, x1000 kcal/mol  [REPLACE] ---
# order within each list matches COMPS = [65, 72, 80, 87, 95]
E_PARTIAL = {                      # overall lipid-lipid partial energy
    "DOPG": [-24.0, -23.0, -22.0, -21.0, -20.0],
    "DOPA": [-25.0, -24.0, -23.0, -22.0, -21.0],
    "DLPG": [-23.0, -22.0, -21.5, -20.5, -19.5],
    "DLPA": [-26.0, -25.0, -24.0, -23.0, -22.0],
}
# --- TOTAL interaction energies, x1000 kcal/mol  [REPLACE] ---
E_TOTAL = {
    "DOPG": [27.0, 28.0, 29.0, 30.0, 31.0],
    "DOPA": [26.0, 27.0, 28.0, 29.0, 30.0],
    "DLPG": [28.0, 29.0, 30.0, 31.0, 32.0],
    "DLPA": [25.5, 26.5, 27.5, 28.5, 29.5],
}
# pure-DOPC reference points (optional; set to None to omit)
E_PARTIAL_DOPC = -22.5
E_TOTAL_DOPC   =  29.0

# --- DECOMPOSED energies (4 panels), x1000 kcal/mol  [REPLACE] ---
# keys: 'sec_sec', 'pc_pc', 'pc_sec', 'overall'
DECOMP_PARTIAL = {
    "sec_sec": {"DOPG": [-1.9, -1.4, -0.9, -0.4, -0.1],
                "DOPA": [-2.1, -1.6, -1.0, -0.5, -0.1],
                "DLPG": [-1.7, -1.3, -0.8, -0.4, -0.1],
                "DLPA": [-2.4, -1.8, -1.2, -0.6, -0.2]},
    "pc_pc":   {"DOPG": [-13.0, -15.0, -17.5, -20.0, -23.0],
                "DOPA": [-12.5, -14.5, -17.0, -19.5, -22.5],
                "DLPG": [-13.5, -15.5, -18.0, -20.5, -23.5],
                "DLPA": [-12.0, -14.0, -16.5, -19.0, -22.0]},
    "pc_sec":  {"DOPG": [-5.5, -4.5, -3.3, -2.1, -0.8],
                "DOPA": [-5.9, -4.8, -3.5, -2.2, -0.9],
                "DLPG": [-5.2, -4.2, -3.1, -2.0, -0.8],
                "DLPA": [-6.3, -5.1, -3.8, -2.4, -1.0]},
    "overall": {"DOPG": [-20.4, -20.9, -21.7, -22.5, -23.9],
                "DOPA": [-20.5, -20.9, -21.5, -22.2, -23.5],
                "DLPG": [-20.4, -21.0, -21.9, -22.9, -24.4],
                "DLPA": [-20.7, -20.9, -21.5, -22.0, -23.2]},
}
DECOMP_TOTAL = {   # [REPLACE]
    "sec_sec": {"DOPG": [17.5, 12.0, 7.5, 3.0, 0.6],
                "DOPA": [8.0, 6.0, 3.5, 1.6, 0.4],
                "DLPG": [11.0, 8.5, 5.0, 2.2, 0.5],
                "DLPA": [12.5, 9.0, 5.5, 2.5, 0.5]},
    "pc_pc":   {"DOPG": [22.0, 24.5, 27.0, 29.0, 30.0],
                "DOPA": [22.2, 24.7, 27.2, 29.2, 30.2],
                "DLPG": [22.1, 24.6, 27.1, 29.1, 30.1],
                "DLPA": [22.3, 24.8, 27.3, 29.3, 30.3]},
    "pc_sec":  {"DOPG": [-5.0, -4.2, -3.0, -1.8, -0.7],
                "DOPA": [-5.4, -4.4, -3.2, -1.9, -0.8],
                "DLPG": [-6.4, -5.2, -3.6, -2.1, -0.9],
                "DLPA": [-5.8, -4.6, -3.3, -2.0, -0.8]},
    "overall": {"DOPG": [34.5, 32.3, 31.5, 30.2, 29.9],
                "DOPA": [24.8, 26.3, 27.5, 28.9, 29.8],
                "DLPG": [26.7, 27.9, 28.5, 29.2, 29.7],
                "DLPA": [29.0, 29.2, 29.5, 29.8, 30.0]},
}

# --- ION-LIPID energies, kcal/mol  [REPLACE] ---
# PA headgroup is a lone hydrogen -> no ion-headgroup bar for DOPA / DLPA (set to None)
ION_HEAD = {"DOPG": [-47.0, -32.0, -22.5, -13.0, -3.0],
            "DLPG": [-11.0, -11.0, -8.0, -6.5, -2.0],
            "DOPA": None,
            "DLPA": None}
ION_PHOS = {"DOPG": [-57.5, -44.5, -29.5, -21.5, -6.0],
            "DLPG": [-63.0, -47.5, -31.0, -23.5, -7.0],
            "DOPA": [-25.0, -20.0, -14.0, -9.0, -3.0],
            "DLPA": [-28.0, -22.0, -15.0, -10.0, -3.5]}

## 3b · Load YOUR simulation results

Three routes — use whichever matches what you have. Each overwrites the placeholder
dictionaries above, so **run one of these, then jump to the figures.**

`load_energies.py` (same folder) does the parsing and the kJ/mol → ×10³ kcal/mol conversion.

### Route A — paste the `gmx energy` terminal output

Paste the `Energy / Average / Err.Est. / RMSD / Tot-Drift` block for each system between
the triple quotes. Add one entry per (lipid, composition).

In [ ]:
from load_energies import (parse_gmx_terminal, parse_xvg, parse_edr,
                           build_energy_tables, load_from_files, kj_to_kcal_k)

TERMINAL = {}          # <-- add one entry per system, then re-run

# TERMINAL[("DOPG", 65)] = """
# Energy                      Average   Err.Est.       RMSD  Tot-Drift
# -------------------------------------------------------------------------------
# Coul-SR:DOPC-DOPC      -1.05e+05         120        890       -12.0  (kJ/mol)
# LJ-SR:DOPC-DOPC        -4.20e+04          60        410        -3.0  (kJ/mol)
# Coul-SR:DOPC-DOPG      -2.30e+04          45        300        -2.0  (kJ/mol)
# LJ-SR:DOPC-DOPG        -9.10e+03          22        150        -1.0  (kJ/mol)
# Coul-SR:DOPG-DOPG      -3.10e+03          18        120        -0.5  (kJ/mol)
# LJ-SR:DOPG-DOPG        -1.20e+03           9         80        -0.2  (kJ/mol)
# """

SOURCES = {k: parse_gmx_terminal(v) for k, v in TERMINAL.items()}
print("parsed systems:", sorted(SOURCES) if SOURCES else "none yet - paste above")

### Route B — point at `.xvg` or `.edr` files

Edit the pattern to match your layout; `{lipid}` and `{comp}` are substituted.

In [ ]:
# SOURCES = load_from_files("energies/{lipid}_{comp}.xvg", LIPIDS)      # .xvg route
# SOURCES = load_from_files("runs/DOPC_{lipid}_{comp}/ener.edr", LIPIDS)  # .edr route (needs panedr)

### Build the tables and overwrite the placeholders

`TOTAL` energies use all pair terms; `PARTIAL` (i–j) energies exclude the like-group
contributions, matching the published definition. Set `PARTIAL_KINDS` to whichever GROMACS
terms correspond to your partial decomposition.

In [ ]:
TOTAL_KINDS   = ("Coul-SR", "LJ-SR")     # all short-range pair terms
PARTIAL_KINDS = ("LJ-SR",)               # <-- adjust to your partial-energy definition

if SOURCES:
    DECOMP_TOTAL   = build_energy_tables(SOURCES, LIPIDS, comps=COMPS, kinds=TOTAL_KINDS)
    DECOMP_PARTIAL = build_energy_tables(SOURCES, LIPIDS, comps=COMPS, kinds=PARTIAL_KINDS)
    E_TOTAL   = DECOMP_TOTAL["overall"]
    E_PARTIAL = DECOMP_PARTIAL["overall"]
    E_TOTAL_DOPC = E_PARTIAL_DOPC = None      # set if you ran a pure-DOPC rerun

    print(f"{'lipid':<7}{'comp':>6}{'sec-sec':>11}{'PC-PC':>11}{'PC-sec':>11}{'overall':>11}")
    for lip in LIPIDS:
        for j, c in enumerate(COMPS):
            r = [DECOMP_TOTAL[k][lip][j] for k in ("sec_sec","pc_pc","pc_sec","overall")]
            if all(v == v for v in r):
                print(f"{lip:<7}{c:>5}%" + "".join(f"{v:11.3f}" for v in r))
    print("\n(units: x10^3 kcal/mol)  -> now re-run the figure cells")
else:
    print("No SOURCES loaded - fill in Route A or B above.")

## 4 · Regression helper (R², slope, t, p)

Matches the published analysis: ordinary least squares of K_A on interaction energy,
reporting R² per composition. Works with or without scipy.

In [ ]:
def linfit(x, y):
    """Return slope, intercept, R2, t_slope, p_two_sided."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)          # drop missing systems
    x, y = x[ok], y[ok]
    n = len(x)
    if n < 3:
        return (np.nan,)*5
    if HAVE_SCIPY:
        s, i, r, p, se = stats.linregress(x, y)
        return s, i, r**2, (s/se if se else np.nan), p
    s, i = np.polyfit(x, y, 1)
    yhat = s*x + i
    r = np.corrcoef(x, y)[0, 1]
    ss = np.sum((y-yhat)**2)
    se = np.sqrt(ss/(n-2)/np.sum((x-x.mean())**2)) if n > 2 else np.nan
    t = s/se if se else np.nan
    # two-sided p from the t distribution via a continued-fraction incomplete beta
    def betacf(a,b,x,itmax=200,eps=3e-7):
        qab,qap,qam=a+b,a+1,a-1; c=1.0; d=1-qab*x/qap
        d=1/(d if abs(d)>1e-30 else 1e-30); h=d
        for m in range(1,itmax+1):
            m2=2*m; aa=m*(b-m)*x/((qam+m2)*(a+m2))
            d=1+aa*d; d=1/(d if abs(d)>1e-30 else 1e-30)
            c=1+aa/(c if abs(c)>1e-30 else 1e-30); h*=d*c
            aa=-(a+m)*(qab+m)*x/((a+m2)*(qap+m2))
            d=1+aa*d; d=1/(d if abs(d)>1e-30 else 1e-30)
            c=1+aa/(c if abs(c)>1e-30 else 1e-30); de=d*c; h*=de
            if abs(de-1)<eps: break
        return h
    def betainc(a,b,x):
        if x<=0: return 0.0
        if x>=1: return 1.0
        from math import lgamma, exp, log
        bt=exp(lgamma(a+b)-lgamma(a)-lgamma(b)+a*log(x)+b*log(1-x))
        return bt*betacf(a,b,x)/a if x<(a+1)/(a+b+2) else 1-bt*betacf(b,a,1-x)/b
    df=n-2
    p = betainc(df/2, 0.5, df/(df+t*t)) if (df>0 and np.isfinite(t)) else np.nan
    return s, i, r**2, t, p

## 5 · Figure 1 & 2 — K_A vs interaction energy

Panel (a) pools all systems; panel (b) splits by composition with a dashed fit and R²,
exactly as in the published figure. Pure DOPC is shown as a filled black circle.

In [ ]:
def ka_vs_energy(E, name, xlabel, e_dopc=None, fname=None, stats_table=True):
    fig = plt.figure(figsize=(13.5, 4.6))
    gsA = fig.add_axes([0.055, 0.13, 0.30, 0.78])          # panel (a)
    # ---- (a) all systems ----
    for lip in LIPIDS:
        if not np.any(np.isfinite(E[lip])): continue
        gsA.errorbar(E[lip], KA[lip], yerr=KA_ERR[lip], fmt=MARKERS[lip],
                     ms=8, color=COLORS[lip], markeredgecolor="k", markeredgewidth=0.6,
                     ecolor="0.6", elinewidth=0.9, capsize=2, lw=0, label=lip)
    if e_dopc is not None:
        gsA.errorbar([e_dopc], [KA_PURE_DOPC[0]], yerr=[KA_PURE_DOPC[1]], fmt="o", ms=9,
                     color="k", ecolor="0.5", capsize=2, label="Pure DOPC")
    gsA.set_xlabel(xlabel); gsA.set_ylabel(r"$K_A$ (mN/m)")
    gsA.text(0.03, 0.95, "(a)", transform=gsA.transAxes, fontsize=13,
             fontweight="bold", va="top")
    gsA.legend(fontsize=8.5, ncol=2, frameon=True)
    gsA.tick_params(direction="in", top=True, right=True)

    # ---- (b) one small panel per composition ----
    left, w, h, gapx, gapy = 0.44, 0.245, 0.34, 0.035, 0.10
    rows, cols = 2, 3
    results = []
    for k, comp in enumerate(COMPS):
        r, c = divmod(k, cols)
        ax = fig.add_axes([left + c*(w+gapx), 0.56 - r*(h+gapy), w, h])
        xs, ys = [], []
        for l in LIPIDS:
            xv, yv = E[l][k], KA[l][k]
            if not (np.isfinite(xv) and np.isfinite(yv)):
                continue
            xs.append(xv); ys.append(yv)
            ax.plot(xv, yv, MARKERS[l], ms=7, color=COLORS[l],
                    markeredgecolor="k", markeredgewidth=0.5)
        if len(xs) < 2:
            ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center",
                    va="center", color="0.6", fontsize=10)
            ax.set_xticks([]); ax.set_yticks([]); continue
        if e_dopc is not None:
            ax.plot(e_dopc, KA_PURE_DOPC[0], "o", ms=7, color="k")
        s, i, R2, t, p = linfit(xs, ys)
        if np.isfinite(s):
            xf = np.linspace(min(xs), max(xs), 50)
            ax.plot(xf, s*xf + i, ls="--", lw=1.3, color="#8B4513")
        ax.text(0.95, 0.93, f"{comp}%", transform=ax.transAxes, ha="right", va="top",
                fontsize=11, fontweight="bold")
        ax.text(0.5, 0.045, ("R$^2$ = %.3f" % R2) if np.isfinite(R2) else "", transform=ax.transAxes,
                ha="center", va="bottom", fontsize=9,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.5))
        ax.tick_params(direction="in", top=True, right=True, labelsize=9)
        if c == 0: ax.set_ylabel(r"$K_A$ (mN/m)", fontsize=11)
        if r == 1 or k >= len(COMPS)-cols: ax.set_xlabel(xlabel, fontsize=10)
        results.append((comp, s, R2, t, p))
    fig.text(0.44, 0.95, "(b)", fontsize=13, fontweight="bold")
    if fname:
        plt.savefig(fname, dpi=600, bbox_inches="tight")
        plt.savefig(fname.replace(".png", ".pdf"), bbox_inches="tight")
    plt.show()

    if stats_table:
        print(f"\n{name}: K_A vs energy regression")
        print(f"{'comp':>6}{'slope':>12}{'R2':>10}{'t':>9}{'p':>10}")
        for comp, s, R2, t, p in results:
            if not np.isfinite(s):
                print(f"{comp:>5}%{'insufficient data':>41}"); continue
            ps = f"{p:.4f}" if np.isfinite(p) else "  n/a"
            print(f"{comp:>5}%{s:12.3f}{R2:10.3f}{t:9.2f}{ps:>10}")
        # pooled fit
        allx = [E[l][k] for l in LIPIDS for k in range(len(COMPS))]
        ally = [KA[l][k] for l in LIPIDS for k in range(len(COMPS))]
        s, i, R2, t, p = linfit(allx, ally)
        if np.isfinite(s):
            ps = f"{p:.4f}" if np.isfinite(p) else "n/a"
            print(f"{'ALL':>6}{s:12.3f}{R2:10.3f}{t:9.2f}{ps:>10}")
    return results

res_partial = ka_vs_energy(E_PARTIAL, "PARTIAL",
    r"Partial interaction energy ($\times$10$^3$ kcal/mol)",
    e_dopc=E_PARTIAL_DOPC, fname="Fig_KA_vs_partial.png")

In [ ]:
res_total = ka_vs_energy(E_TOTAL, "TOTAL",
    r"Total interaction energy ($\times$10$^3$ kcal/mol)",
    e_dopc=E_TOTAL_DOPC, fname="Fig_KA_vs_total.png")

## 6 · Figure 3 & 4 — decomposed interaction energies (4 panels)

Grouped bars, one panel per pair type: (a) secondary–secondary, (b) DOPC–DOPC,
(c) DOPC–secondary, (d) overall lipid–lipid. Composition on the x-axis, as published.

In [ ]:
PANEL_TITLES = {"sec_sec": "secondary–secondary", "pc_pc": "DOPC–DOPC",
                "pc_sec": "DOPC–secondary",       "overall": "overall lipid–lipid"}

def decomposed_figure(DEC, fname, ylabel=r"Interaction energy ($\times$10$^3$ kcal/mol)"):
    keys = ["sec_sec", "pc_pc", "pc_sec", "overall"]
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.4))
    x = np.arange(len(COMPS)); n = len(LIPIDS); width = 0.8/n
    for ax, key, tag in zip(axes.flat, keys, "abcd"):
        for i, lip in enumerate(LIPIDS):
            ax.bar(x + (i-(n-1)/2)*width, DEC[key][lip], width, label=lip,
                   color=COLORS[lip], edgecolor="black", linewidth=0.7)
        ax.set_xticks(x); ax.set_xticklabels([f"{c}%" for c in COMPS])
        ax.axhline(0, color="k", lw=0.8)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.text(0.02, 0.97, f"({tag})", transform=ax.transAxes, fontsize=13,
                fontweight="bold", va="top")
        ax.text(0.5, 1.02, PANEL_TITLES[key], transform=ax.transAxes,
                ha="center", fontsize=11)
        ax.tick_params(direction="in", top=True, right=True)
        ax.xaxis.set_ticks_position("bottom")
        ax.legend(fontsize=9, ncol=2, frameon=True)
    plt.tight_layout()
    plt.savefig(fname, dpi=600, bbox_inches="tight")
    plt.savefig(fname.replace(".png", ".pdf"), bbox_inches="tight")
    plt.show()

decomposed_figure(DECOMP_PARTIAL, "Fig_decomposed_partial.png")

In [ ]:
decomposed_figure(DECOMP_TOTAL, "Fig_decomposed_total.png")

## 7 · Figure 5 — ion–lipid interaction energies

Solid bars = ion–headgroup, hatched bars = ion–phosphate. DOPA and DLPA have a lone
hydrogen as the PA headgroup, so they carry no solid bar (as for DSPA in the published
figure).

In [ ]:
def ion_lipid_figure(fname="Fig_ion_lipid.png"):
    fig, ax = plt.subplots(figsize=(8.6, 5.6))
    x = np.arange(len(COMPS)); n = len(LIPIDS); width = 0.8/n
    for i, lip in enumerate(LIPIDS):
        xpos = x + (i-(n-1)/2)*width
        head, phos = ION_HEAD.get(lip), ION_PHOS.get(lip)
        if head is not None:                       # solid = ion-headgroup
            ax.bar(xpos - width*0.2, head, width*0.6, color=COLORS[lip],
                   edgecolor="black", linewidth=0.7)
        if phos is not None:                       # hatched = ion-phosphate
            ax.bar(xpos + width*0.2, phos, width*0.6, color=COLORS[lip],
                   edgecolor="black", linewidth=0.7, hatch="///")
    ax.set_xticks(x); ax.set_xticklabels([f"{c}%" for c in COMPS])
    ax.set_xlabel("Molar ratio of DOPC (mol %)")
    ax.set_ylabel("Interaction energy (kcal/mol)")
    ax.axhline(0, color="k", lw=0.8)
    ax.tick_params(direction="in", top=True, right=True)
    lipid_h = [Patch(facecolor=COLORS[l], edgecolor="k", label=l) for l in LIPIDS]
    type_h  = [Patch(facecolor="white", edgecolor="k", label="ion–headgroup"),
               Patch(facecolor="white", edgecolor="k", hatch="///", label="ion–phosphate")]
    leg1 = ax.legend(handles=lipid_h, loc="lower left", fontsize=9, ncol=2, frameon=True)
    ax.add_artist(leg1)
    ax.legend(handles=type_h, loc="lower right", fontsize=9, frameon=True)
    plt.tight_layout()
    plt.savefig(fname, dpi=600, bbox_inches="tight")
    plt.savefig(fname.replace(".png", ".pdf"), bbox_inches="tight")
    plt.show()

ion_lipid_figure()

## 8 · Getting the energies out of GROMACS

For each system, the rerun (`energygrps = DOPC <secondary>`, `mdrun -rerun ... -nb cpu`)
writes an `ener.edr` containing the pair terms. Extract and convert:

```bash
gmx energy -f ener.edr -o inter.xvg     # pick Coul-SR:*, LJ-SR:* for each pair
```

```python
def read_xvg(path):
    leg, rows = [], []
    for line in open(path):
        if line.startswith("@") and " legend " in line: leg.append(line.split('"')[1])
        elif not line.startswith(("#", "@")): rows.append([float(v) for v in line.split()])
    return leg, np.array(rows)

leg, d = read_xvg("inter.xvg")
for i, name in enumerate(leg):
    mean_kJ = d[:, i+1].mean()
    print(f"{name:<26}{mean_kJ:10.1f} kJ/mol = {kj_to_kcal_k(mean_kJ):8.3f} x1e3 kcal/mol")
```

**Conventions to keep consistent with the published figures**

* *Partial* (i–j) energies exclude head–head, phosphate–phosphate, glycerol–glycerol and
  tail–tail pairs between different lipids; *total* energies include all pairs.
* Coul-SR is the real-space Coulomb term only (the PME reciprocal part is not
  group-decomposable), so keep cutoffs identical across all systems.
* Report per-pair sums in ×10³ kcal/mol; partial energies come out mostly negative and
  total energies positive, as in the source figures.
